# Rotation optimization features

This notebook demonstrates fixed z-axis rotations, fixed Wigner rotations, beta-only Wigner optimization, full Euler-angle Wigner optimization, and the diagnostic that flags possible redundancy with orbital phase alignment.

In [ ]:
import numpy as np

from waveformtools.comparison import AlignmentSpec, RotationSpec, mode_match, rotate_modes
from waveformtools.modes_array import ModesArray


def make_full_ell2_modes(time_axis=None):
    if time_axis is None:
        time_axis = np.linspace(-6.0, 6.0, 160)
    envelope = np.exp(-0.04 * time_axis**2)
    modes = ModesArray(ell_max=2, time_axis=time_axis, spin_weight=-2)
    modes.create_modes_array(ell_max=2, data_len=len(time_axis))
    for index, emm in enumerate(range(-2, 3), start=1):
        amplitude = 0.15 + 0.12 * index
        frequency = 0.25 + 0.08 * index
        phase = 0.17 * index
        modes.set_mode_data(
            ell=2,
            emm=emm,
            data=amplitude * envelope * np.exp(1j * (frequency * time_axis + phase)),
        )
    return modes


reference = make_full_ell2_modes()


## Fixed z-axis and fixed Wigner rotations

In [ ]:
z_rotated = rotate_modes(reference, RotationSpec(kind="z_axis", angle=0.3))
wigner_rotated = rotate_modes(reference, RotationSpec(kind="wigner", alpha=0.3, beta=0.1, gamma=-0.2))

print("fixed z-axis mismatch=", mode_match(reference, z_rotated, time_alignment="none", phase_alignment="none").mismatch)
print("fixed Wigner mismatch=", mode_match(reference, wigner_rotated, time_alignment="none", phase_alignment="none").mismatch)


## Beta-only Wigner optimization

In [ ]:
candidate = rotate_modes(reference, RotationSpec(kind="wigner", beta=0.25))

result = mode_match(
    reference,
    candidate,
    time_alignment="none",
    phase_alignment="none",
    rotation=RotationSpec(
        kind="wigner",
        optimize_parameters=("beta",),
        parameter_bounds={"beta": (-0.5, 0.1)},
    ),
)

print("match=", result.match)
print("mismatch=", result.mismatch)
print("best rotation=", result.best_parameters["rotation"])
print("optimizer=", result.optimizer)


## Full Euler-angle Wigner optimization

In [ ]:
candidate = rotate_modes(reference, RotationSpec(kind="wigner", alpha=0.12, beta=0.18, gamma=-0.09))

result = mode_match(
    reference,
    candidate,
    time_alignment="none",
    phase_alignment="none",
    rotation=RotationSpec(
        kind="wigner",
        optimize_parameters=("alpha", "beta", "gamma"),
        parameter_bounds={
            "alpha": (-0.1, 0.2),
            "beta": (-0.3, 0.0),
            "gamma": (-0.2, 0.1),
        },
    ),
)

print("match=", result.match)
print("mismatch=", result.mismatch)
print("best rotation=", result.best_parameters["rotation"])
print("rotation diagnostics=", result.diagnostics["rotation_optimization"])


## Redundancy diagnostic with orbital phase alignment

The optimizer allows `alpha` or `gamma` together with orbital phase alignment. The result includes a diagnostic flag because these freedoms can be correlated.

In [ ]:
candidate = rotate_modes(reference, RotationSpec(kind="wigner", beta=0.1))

result = mode_match(
    reference,
    candidate,
    time_alignment="none",
    phase_alignment="orbital_phase_and_global",
    alignment=AlignmentSpec(orbital_phase_samples=721),
    rotation=RotationSpec(
        kind="wigner",
        optimize_parameters=("alpha", "beta"),
        parameter_bounds={"alpha": (-0.2, 0.2), "beta": (-0.2, 0.0)},
    ),
)

print("match=", result.match)
print("mismatch=", result.mismatch)
print("phase_degeneracy_possible=", result.diagnostics["rotation_optimization"]["phase_degeneracy_possible"])
